In [19]:
# import zipfile
# import glob
# import numpy as np
# import os

# years = list(np.arange(2020,2026))
# folder = "/home/student/rishi/epa_data/epa_data"

# for f in glob.glob(f"{folder}/*.zip"):
#     if any(f.endswith(f"{year}.zip") for year in years):
#         with zipfile.ZipFile(f, "r") as z:
#             for member in z.infolist():
#                 member.filename = os.path.basename(member.filename)
#                 if member.filename:
#                     z.extract(member, "output_folder/")

In [6]:
import pandas as pd
import os

types=['hourly_42401',
 'hourly_88101',
 'hourly_44201',
 'hourly_42602',
 'hourly_42101',
 'hourly_81102',
]

In [21]:
def separate_and_filter(df, pol):
    df.columns = df.columns.str.strip().str.replace("_", " ")
    df["Timestamp"] = pd.to_datetime(df["Date GMT"] + " " + df["Time GMT"], format="%Y-%m-%d %H:%M")
    s=df['Parameter Name'].iloc[0]+' ' +df['Units of Measure'].iloc[0]
    year=pd.to_datetime(df["Date Local"].iloc[0]).year
    df[s]=df['Sample Measurement']
    df["Key"]=df.apply(lambda x: (x['State Code'],x['County Code'], x['Site Num']), axis=1)
    df_f=df[['Key','Timestamp', s]]
    output_dir = "epa_data_by_site"
    os.makedirs(output_dir, exist_ok=True)

    for site_id, group_df in df_f.groupby("Key"):
        safe_name = f"site_{site_id[0]}_{site_id[1]}_{site_id[2]}_{pol}_{year}"
        safe_name = safe_name.replace("/", "_").replace(" ", "_")
        group_df.drop(columns=["Key"]).to_csv(os.path.join(output_dir, f"{safe_name}.csv"), index=False)
    print("Saved", pol)

In [10]:
base="/mnt/2TB_WD/rishi/EPA"
years = [2022,2023,2024, 2025]
output_dir = os.path.join(base,"epa_data_by_site")
os.makedirs(output_dir, exist_ok=True)
os.makedirs("EPA_sites_separate_pols", exist_ok=True)

In [23]:
type_to_pol = {
    'hourly_42401': 'Sulphur Dioxide',
    'hourly_88101': 'PM2.5',
    'hourly_44201': 'Ozone',
    'hourly_42602': 'Nitrogen Dioxide',
    'hourly_42101': 'Carbon Monoxide',
    'hourly_81102': 'PM10',
}

for year in years:
    for type in types:
        path = fr"{base}/{type}_{year}.csv"
        if not os.path.exists(path):
            print(f"Skipping {path} (not found)")
            continue
        df = pd.read_csv(path, low_memory=False)
        try:
            separate_and_filter(df, type_to_pol[type])
        except Exception as e:
            print("Failed on :", e, path)

Saved Sulphur Dioxide
Saved PM2.5
Saved Ozone
Saved Nitrogen Dioxide
Saved Carbon Monoxide
Saved PM10
Saved Sulphur Dioxide
Saved PM2.5
Saved Ozone
Saved Nitrogen Dioxide
Saved Carbon Monoxide
Saved PM10
Saved Sulphur Dioxide
Saved PM2.5
Saved Ozone
Saved Nitrogen Dioxide
Saved Carbon Monoxide
Saved PM10
Saved Sulphur Dioxide
Saved PM2.5
Saved Ozone
Saved Nitrogen Dioxide
Saved Carbon Monoxide
Saved PM10


In [11]:
import glob
from collections import defaultdict

pols = ["PM2.5 (µg/m³)", "PM10 (µg/m³)", "NO2 (µg/m³)", "SO2 (µg/m³)", "CO (mg/m³)", "Ozone (µg/m³)"]
original_units = {
    "Sulphur Dioxide":    "Sulfur dioxide Parts per billion",#2.62
    "PM2.5":              "PM2.5 - Local Conditions Micrograms/cubic meter (LC)",
    "Ozone":              "Ozone Parts per million",#1.96 *1000
    "Nitrogen Dioxide":   "Nitrogen dioxide (NO2) Parts per billion", #1.88
    "Carbon Monoxide":    "Carbon monoxide Parts per million",#1.15
    "PM10":               "PM10 Total 0-10um STP Micrograms/cubic meter (25 C)",
}
conversion_factors = {
    "Sulphur Dioxide":  2.62,
    "PM2.5":            1.0,
    "Ozone":            1.96 * 1000,
    "Nitrogen Dioxide": 1.88,
    "Carbon Monoxide":  1.15,
    "PM10":             1.0,
}
who_names = {
    "Sulphur Dioxide":  "SO2 (µg/m³)",
    "PM2.5":            "PM2.5 (µg/m³)",
    "Ozone":            "Ozone (µg/m³)",
    "Nitrogen Dioxide": "NO2 (µg/m³)",
    "Carbon Monoxide":  "CO (mg/m³)",
    "PM10":             "PM10 (µg/m³)",
}
units_to_pol = {v: k for k, v in original_units.items()}

TARGET_START = '2023-01-01'
TARGET_END   = '2025-12-31 23:00'
target_index = pd.date_range(start=TARGET_START, end=TARGET_END, freq='h')

# Group per-year CSVs by (site, pollutant)
groups = defaultdict(list)
for fpath in glob.glob(os.path.join(output_dir, "*.csv")):
    fname = os.path.basename(fpath).replace(".csv", "")
    parts = fname.split("_")
    key = "_".join(parts[:-1])
    groups[key].append((int(parts[-1]), fpath))

for key, file_list in groups.items():
    file_list.sort(key=lambda x: x[0])
    dfs = [pd.read_csv(fp) for _, fp in file_list]
    combined = pd.concat(dfs, ignore_index=True)

    combined["Timestamp"] = pd.to_datetime(combined["Timestamp"])
    combined = combined.drop_duplicates(subset="Timestamp").set_index("Timestamp").sort_index()
    combined = combined.reindex(target_index)
    combined.index.name = "Timestamp"

    # Detect pollutant from column name, convert and rename
    orig_col = next((c for c in combined.columns if c in units_to_pol), None)
    if orig_col is None:
        print(f"Could not detect pollutant column for key: {key}")
        continue
    pol_name = units_to_pol[orig_col]
    combined[orig_col] = combined[orig_col] * conversion_factors[pol_name]
    combined = combined.rename(columns={orig_col: who_names[pol_name]})

    # Replace pollutant suffix in key with chemical formula (e.g. Sulphur_Dioxide -> SO2)
    pol_suffix = pol_name.replace(" ", "_")
    site_prefix = key[: -len(pol_suffix) - 1]
    formula = who_names[pol_name].split(" ")[0]
    combined.to_csv(os.path.join("EPA_sites_separate_pols", f"{site_prefix}_{formula}.csv"))

print(f"Saved {len(groups)} multi-year files to joined/")

Saved 4020 multi-year files to joined/
